# vehicle_model 使用样例

演示 4 种模型×积分器组合的轨迹差异：
1. 运动学 + 欧拉
2. 运动学 + RK4
3. 动力学 + 欧拉
4. 动力学 + RK4

每种组合使用相同的动作序列，对比轨迹和速度曲线。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sim_env.vehicle_model import (
    VehicleModel, VehicleModelConfig, VehicleParams, VehicleState,
    ModelType, IntegratorType,
)

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
%matplotlib inline

In [ ]:
# 仿真参数
DT = 0.05        # 时间步长 (s)
N_STEPS = 50    # 总步数 → 20s
V_INIT = 10.0    # 初始速度 (m/s)

# 动作序列：前半段加速+左转，后半段减速+右转
def make_actions(n):
    actions = np.zeros((n, 2))
    half = n // 2
    actions[:half, 0] = 2.0    # 加速 2 m/s²
    actions[:half, 1] = 0.8    # 左转 0.3 rad/s
    actions[half:, 0] = -1.0   # 减速 -1 m/s²
    actions[half:, 1] = -0.2   # 右转 -0.2 rad/s
    return actions

ACTIONS = make_actions(N_STEPS)

In [ ]:
def simulate(model_type, integrator, label):
    """运行一次仿真，返回轨迹数据。"""
    vm = VehicleModel(cfg=VehicleModelConfig(
        vehicle_params=VehicleParams(),
        model_type=model_type,
        integrator_type=integrator,
    ))
    vm.reset(x=0, y=0, theta=0, v=V_INIT)

    xs, ys, thetas, vs, ts = [0.0], [0.0], [0.0], [V_INIT], [0.0]
    for i in range(N_STEPS):
        s = vm.step(ACTIONS[i], DT)
        xs.append(s.x)
        ys.append(s.y)
        thetas.append(s.theta)
        vs.append(s.v)
        ts.append((i + 1) * DT)

    return {
        "label": label,
        "x": np.array(xs), "y": np.array(ys),
        "theta": np.array(thetas), "v": np.array(vs),
        "t": np.array(ts),
    }

## 示例 1：运动学模型 — 欧拉 vs RK4

运动学模型中 `ω` 直接作为 `dθ/dt`，不涉及转向角积分。
对比欧拉和 RK4 在相同动作下的轨迹差异。

In [ ]:
kin_euler = simulate(ModelType.KINEMATIC, IntegratorType.EULER, "运动学+欧拉")
kin_rk4   = simulate(ModelType.KINEMATIC, IntegratorType.RK4,   "运动学+RK4")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 轨迹
ax = axes[0]
for d, c in [(kin_euler, "tab:blue"), (kin_rk4, "tab:orange")]:
    ax.plot(d["x"], d["y"], color=c, label=d["label"])
    ax.plot(d["x"][0], d["y"][0], "o", color=c, ms=8)
    ax.plot(d["x"][-1], d["y"][-1], "s", color=c, ms=8)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("轨迹对比")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)

# 速度
ax = axes[1]
for d, c in [(kin_euler, "tab:blue"), (kin_rk4, "tab:orange")]:
    ax.plot(d["t"], d["v"], color=c, label=d["label"])
ax.set_xlabel("时间 (s)")
ax.set_ylabel("速度 (m/s)")
ax.set_title("速度曲线")
ax.legend()
ax.grid(True, alpha=0.3)

# 朝向角
ax = axes[2]
for d, c in [(kin_euler, "tab:blue"), (kin_rk4, "tab:orange")]:
    ax.plot(d["t"], np.degrees(d["theta"]), color=c, label=d["label"])
ax.set_xlabel("时间 (s)")
ax.set_ylabel("朝向角 (°)")
ax.set_title("朝向角曲线")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle("运动学模型：欧拉 vs RK4", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 示例 2：动力学模型 — 欧拉 vs RK4

动力学模型中 `ω` 积分为前轮转向角 `δ`，`dθ/dt = v·tan(δ)/L`，且含空气阻力。
高速时阻力效应明显，RK4 精度优势更大。

In [ ]:
dyn_euler = simulate(ModelType.DYNAMIC, IntegratorType.EULER, "动力学+欧拉")
dyn_rk4   = simulate(ModelType.DYNAMIC, IntegratorType.RK4,   "动力学+RK4")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
for d, c in [(dyn_euler, "tab:green"), (dyn_rk4, "tab:red")]:
    ax.plot(d["x"], d["y"], color=c, label=d["label"])
    ax.plot(d["x"][0], d["y"][0], "o", color=c, ms=8)
    ax.plot(d["x"][-1], d["y"][-1], "s", color=c, ms=8)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("轨迹对比")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
for d, c in [(dyn_euler, "tab:green"), (dyn_rk4, "tab:red")]:
    ax.plot(d["t"], d["v"], color=c, label=d["label"])
ax.set_xlabel("时间 (s)")
ax.set_ylabel("速度 (m/s)")
ax.set_title("速度曲线（含空气阻力）")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[2]
for d, c in [(dyn_euler, "tab:green"), (dyn_rk4, "tab:red")]:
    ax.plot(d["t"], np.degrees(d["theta"]), color=c, label=d["label"])
ax.set_xlabel("时间 (s)")
ax.set_ylabel("朝向角 (°)")
ax.set_title("朝向角曲线")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle("动力学模型：欧拉 vs RK4", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 示例 3：运动学 vs 动力学（同一积分器 RK4）

固定使用 RK4 积分器，对比运动学和动力学模型的行为差异：
- 运动学：无阻力，`ω` 直接控制转向
- 动力学：有空气阻力，`ω` 积分为转向角后间接影响转向

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
for d, c in [(kin_rk4, "tab:orange"), (dyn_rk4, "tab:red")]:
    ax.plot(d["x"], d["y"], color=c, label=d["label"])
    ax.plot(d["x"][0], d["y"][0], "o", color=c, ms=8)
    ax.plot(d["x"][-1], d["y"][-1], "s", color=c, ms=8)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("轨迹对比")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
for d, c in [(kin_rk4, "tab:orange"), (dyn_rk4, "tab:red")]:
    ax.plot(d["t"], d["v"], color=c, label=d["label"])
ax.set_xlabel("时间 (s)")
ax.set_ylabel("速度 (m/s)")
ax.set_title("速度曲线")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[2]
for d, c in [(kin_rk4, "tab:orange"), (dyn_rk4, "tab:red")]:
    ax.plot(d["t"], np.degrees(d["theta"]), color=c, label=d["label"])
ax.set_xlabel("时间 (s)")
ax.set_ylabel("朝向角 (°)")
ax.set_title("朝向角曲线")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle("运动学 vs 动力学（RK4 积分器）", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 示例 4：四种组合总览

将 4 种组合的轨迹画在同一张图上，直观对比。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

all_runs = [
    (kin_euler, "tab:blue",   "-"),
    (kin_rk4,   "tab:orange", "-"),
    (dyn_euler, "tab:green",  "--"),
    (dyn_rk4,   "tab:red",    "--"),
]

for d, c, ls in all_runs:
    ax.plot(d["x"], d["y"], color=c, ls=ls, lw=2, label=d["label"])
    ax.plot(d["x"][0], d["y"][0], "o", color=c, ms=8)
    ax.plot(d["x"][-1], d["y"][-1], "s", color=c, ms=8)

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("四种模型×积分器组合轨迹总览")
ax.set_aspect("equal")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()